# Review Full Dataset Data Sources

Use this notebook to formally inspect the external data sources for the full `fake_mantis_invest` universe. The goal here is not only to fetch the data, but to dereference URLs and filing paths, extract preview text, and judge how useful each source will be for the long-term product.


## Scope

- Build the full ticker universe from the fake Mantis dataset.
- Fetch official SEC company mappings and filing metadata.
- Build full-document URLs for latest 10-K / 10-Q filings.
- Inflate a sample of those filing URLs into extracted text previews.
- Fetch macro context from FRED.
- Fetch company profile enrichment from FMP.
- Fetch broad news metadata from GDELT.
- Inflate a sample of news URLs into extracted text previews.
- Create a source-review table describing what each source is useful for.

### Helpful Short Forms

- `SEC` = **U.S. Securities and Exchange Commission**
- `CIK` = **Central Index Key**, the SEC company identifier
- `FRED` = **Federal Reserve Economic Data**
- `GDELT` = **Global Database of Events, Language, and Tone**
- `FMP` = **Financial Modeling Prep**
- `10-K` = annual filing
- `10-Q` = quarterly filing
- `8-K` = current event filing
- `MD&A` = **Management's Discussion and Analysis**


In [1]:
from __future__ import annotations

import gzip
import json
import os
import re
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
REVIEW_DIR = INTERIM_DIR / 'source_reviews'
load_dotenv(ROOT / '.env')

for path in [
    EXTERNAL_DIR / 'sec',
    EXTERNAL_DIR / 'fred',
    EXTERNAL_DIR / 'gdelt',
    EXTERNAL_DIR / 'fmp',
    REVIEW_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

SEC_HEADERS = {
    'User-Agent': 'portfolio-analyzer research contact@example.com',
    'Accept-Encoding': 'gzip, deflate',
}

MAX_SEC_DOCS_TO_INFLATE = 10
MAX_GDELT_ARTICLES_PER_TICKER = 3
MAX_NEWS_URLS_TO_INFLATE = 10

def read_response_bytes(response) -> bytes:
    payload = response.read()
    encoding = (response.headers.get('Content-Encoding') or '').lower()
    if 'gzip' in encoding:
        payload = gzip.decompress(payload)
    return payload

def fetch_url(url: str, headers: dict[str, str] | None = None, *, retries: int = 4, base_delay: float = 1.5) -> bytes:
    merged_headers = headers or {}
    last_error = None
    for attempt in range(retries):
        request = Request(url, headers=merged_headers)
        try:
            with urlopen(request) as response:
                return read_response_bytes(response)
        except HTTPError as error:
            last_error = error
            if error.code != 429 or attempt == retries - 1:
                raise
            retry_after = error.headers.get('Retry-After')
            delay = float(retry_after) if retry_after else base_delay * (2 ** attempt)
            print(f'Rate limited for {url}. Sleeping {delay:.1f}s before retry {attempt + 2}/{retries}.')
            time.sleep(delay)
    raise last_error if last_error else RuntimeError('Request failed without a captured HTTPError')

def fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:
    return json.loads(fetch_url(url, headers=headers).decode('utf-8'))

def fetch_text(url: str, headers: dict[str, str] | None = None) -> str:
    return fetch_url(url, headers=headers).decode('utf-8', errors='ignore')

def html_to_text(html: str) -> str:
    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)
    no_tags = re.sub(r'<[^>]+>', ' ', no_script)
    return re.sub(r'\s+', ' ', no_tags).strip()

def simple_chunk(text: str, chunk_size: int = 1800) -> list[str]:
    clean = re.sub(r'\s+', ' ', text).strip()
    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]

def build_primary_doc_url(cik: str, accession_number: str, primary_document: str) -> str:
    accession = accession_number.replace('-', '')
    return f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary_document}'

transactions = pd.read_csv(RAW_DIR / 'fake_mantis_invest.csv')
ticker_universe = sorted({
    str(value).strip()
    for value in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist()
    if str(value).strip()
})
len(ticker_universe), ticker_universe[:10]


(12,
 ['AAPL',
  'AMZN',
  'AVGO',
  'BRK.B',
  'GOOGL',
  'META',
  'MSFT',
  'NVDA',
  'QQQ',
  'TSLA'])

## 1. Full Ticker Universe

Build the full stock / ETF universe from the fake Mantis dataset so every later source review is grounded in the actual portfolio universe.


In [2]:
ticker_overview = pd.DataFrame({'ticker': ticker_universe})
ticker_overview['is_blank'] = ticker_overview['ticker'].eq('')
ticker_overview.head(20)


,ticker,is_blank
0,AAPL,False
1,AMZN,False
2,AVGO,False
3,BRK.B,False
4,GOOGL,False
5,META,False
6,MSFT,False
7,NVDA,False
8,QQQ,False
9,TSLA,False


## 2. SEC Company Master + Filing Metadata For Full Universe

This creates the official SEC mapping from ticker to CIK, then collects the recent filing metadata for every mapped company.


In [3]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'
company_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)
company_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})
company_master['ticker'] = company_master['ticker'].astype(str).str.upper()
company_master['cik'] = company_master['cik'].astype(str).str.zfill(10)
company_master = company_master[company_master['ticker'].isin(ticker_universe)].copy()
company_master.to_csv(EXTERNAL_DIR / 'sec' / 'company_master_full_universe.csv', index=False)
company_master.head(20)


,cik,ticker,company_name
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
5,0001730168,AVGO,Broadcom Inc.
6,0001326801,META,"Meta Platforms, Inc."
7,0001318605,TSLA,"Tesla, Inc."
14,0001403161,V,VISA INC.
51,0001067839,QQQ,"INVESCO QQQ TRUST, SERIES 1"


In [4]:
filing_frames = []
for row in company_master.itertuples(index=False):
    submissions_url = f'https://data.sec.gov/submissions/CIK{row.cik}.json'
    submissions = fetch_json(submissions_url, headers=SEC_HEADERS)
    recent = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))
    if recent.empty:
        continue
    recent = recent[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].copy()
    recent['ticker'] = row.ticker
    recent['company_name'] = row.company_name
    recent['cik'] = row.cik
    recent['primary_document_url'] = recent.apply(
        lambda r: build_primary_doc_url(row.cik, r['accessionNumber'], r['primaryDocument']),
        axis=1,
    )
    filing_frames.append(recent)

filings_full = pd.concat(filing_frames, ignore_index=True) if filing_frames else pd.DataFrame()
filings_full.to_csv(EXTERNAL_DIR / 'sec' / 'filings_full_universe.csv', index=False)
filings_full.head(20)


,accessionNumber,filingDate,form,primaryDocument,ticker,company_name,cik,primary_document_url
0,0000102909-26-000426,2026-03-26,SCHEDULE 13G/A,xslSCHEDULE_13G_X02/primary_doc.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
1,0001199039-26-000003,2026-03-24,4,xslF345X06/wk-form4_1774386816.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
2,0001725292-26-000002,2026-03-20,4,xslF345X06/wk-form4_1774052040.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
3,0001526111-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051982.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
4,0001696841-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051862.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
5,0001283854-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051812.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
6,0001347842-26-000011,2026-03-20,4,xslF345X06/wk-form4_1774051696.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
7,0001588670-26-000010,2026-03-20,4,xslF345X06/wk-form4_1774051632.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
8,0001197649-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051558.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
9,0001628280-26-020268,2026-03-20,144,xsl144X01/primary_doc.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...


## 3. Latest 10-K / 10-Q URLs + SEC Company Facts Snapshot

This step narrows the SEC filing list to the latest annual / quarterly filing per ticker and also previews official company facts.


In [5]:
latest_reporting_filings = (
    filings_full[filings_full['form'].isin(['10-K', '10-Q'])]
    .sort_values(['ticker', 'filingDate'], ascending=[True, False])
    .drop_duplicates(subset=['ticker'], keep='first')
    .reset_index(drop=True)
)
latest_reporting_filings.to_csv(EXTERNAL_DIR / 'sec' / 'latest_reporting_filings.csv', index=False)
latest_reporting_filings[['ticker', 'form', 'filingDate', 'primaryDocument', 'primary_document_url']].head(20)


,ticker,form,filingDate,primaryDocument,primary_document_url
0,AAPL,10-Q,2026-01-30,aapl-20251227.htm,https://www.sec.gov/Archives/edgar/data/320193...
1,AMZN,10-K,2026-02-06,amzn-20251231.htm,https://www.sec.gov/Archives/edgar/data/101872...
2,AVGO,10-Q,2026-03-11,avgo-20260201.htm,https://www.sec.gov/Archives/edgar/data/173016...
3,GOOGL,10-K,2026-02-05,goog-20251231.htm,https://www.sec.gov/Archives/edgar/data/165204...
4,META,10-K,2026-01-29,meta-20251231.htm,https://www.sec.gov/Archives/edgar/data/132680...
5,MSFT,10-Q,2026-01-28,msft-20251231.htm,https://www.sec.gov/Archives/edgar/data/789019...
6,NVDA,10-K,2026-02-25,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/104581...
7,TSLA,10-K,2026-01-29,tsla-20251231.htm,https://www.sec.gov/Archives/edgar/data/131860...
8,V,10-Q,2026-01-30,v-20251231.htm,https://www.sec.gov/Archives/edgar/data/140316...


In [6]:
company_fact_rows = []
for row in company_master.itertuples(index=False):
    facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{row.cik}.json'
    try:
        company_facts = fetch_json(facts_url, headers=SEC_HEADERS)
        us_gaap = company_facts.get('facts', {}).get('us-gaap', {})
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': 'available',
            'fact_tag_count': len(us_gaap),
            'sample_fact_tags': ', '.join(list(us_gaap.keys())[:10]),
        })
    except HTTPError as error:
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': f'http_{error.code}',
            'fact_tag_count': 0,
            'sample_fact_tags': '',
        })

company_facts_review = pd.DataFrame(company_fact_rows)
company_facts_review.to_csv(EXTERNAL_DIR / 'sec' / 'company_facts_review.csv', index=False)
company_facts_review.head(20)

,ticker,company_name,cik,facts_status,fact_tag_count,sample_fact_tags
0,NVDA,NVIDIA CORP,0001045810,available,625,"AcceleratedShareRepurchaseProgramAdjustment, A..."
1,AAPL,Apple Inc.,0000320193,available,503,"AccountsPayable, AccountsPayableCurrent, Accou..."
2,GOOGL,Alphabet Inc.,0001652044,available,521,"AccountsPayableCurrent, AccountsReceivableNetC..."
3,MSFT,MICROSOFT CORP,0000789019,available,543,"AccountsPayableCurrent, AccountsReceivableNet,..."
4,AMZN,AMAZON COM INC,0001018724,available,538,"AccountsPayable, AccountsPayableCurrent, Accou..."
5,AVGO,Broadcom Inc.,0001730168,available,432,"AccountsPayableCurrent, AccountsReceivableNetC..."
6,META,"Meta Platforms, Inc.",0001326801,available,455,AccountsPayableAndOtherAccruedLiabilitiesCurre...
7,TSLA,"Tesla, Inc.",0001318605,available,638,"AccountsAndNotesReceivableNet, AccountsPayable..."
8,V,VISA INC.,0001403161,available,625,"AccountsPayable, AccountsPayableCurrent, Accou..."
9,QQQ,"INVESCO QQQ TRUST, SERIES 1",0001067839,http_404,0,


## 4. Inflate SEC Filing URLs Into Text Preview

This is where we move from metadata into actual filing content. We only inflate a limited sample so the notebook stays practical and avoids hammering SEC.


In [15]:
latest_reporting_filings['primary_document_url'].head(1)

0    https://www.sec.gov/Archives/edgar/data/320193...
Name: primary_document_url, dtype: object

In [7]:
sec_doc_reviews = []
for row in latest_reporting_filings.head(MAX_SEC_DOCS_TO_INFLATE).itertuples(index=False):
    html = fetch_text(row.primary_document_url, headers=SEC_HEADERS)
    text = html_to_text(html)
    chunks = simple_chunk(text)
    preview = chunks[0] if chunks else ''
    sec_doc_reviews.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'primary_document_url': row.primary_document_url,
        'document_char_count': len(text),
        'chunk_count': len(chunks),
        'contains_risk_word': 'risk' in text.lower(),
        'contains_management_word': 'management' in text.lower(),
        'text_preview': preview[:1200],
    })

sec_doc_review_df = pd.DataFrame(sec_doc_reviews)
sec_doc_review_df.to_csv(REVIEW_DIR / 'sec_document_review.csv', index=False)
sec_doc_review_df[['ticker', 'form', 'document_char_count', 'chunk_count', 'contains_risk_word', 'contains_management_word', 'text_preview']].head(10)


,ticker,form,document_char_count,chunk_count,contains_risk_word,contains_management_word,text_preview
0,AAPL,10-Q,72361,41,True,True,aapl-20251227 false 2026 Q1 0000320193 --09-26...
1,AMZN,10-K,326605,182,True,True,amzn-20251231 false 2025 FY 0001018724 P4Y0M P...
2,AVGO,10-Q,203589,114,True,True,avgo-20260201 0001730168 November 1 2026 Q1 FA...
3,GOOGL,10-K,398500,222,True,True,goog-20251231 FALSE 2025 FY 0001652044 P7Y P2Y...
4,META,10-K,549559,306,True,True,meta-20251231 false 2025 FY 0001326801 P5Y P1Y...
5,MSFT,10-Q,312016,174,True,True,10-Q --06-30 0000789019 Q2 false 2026 http://f...
6,NVDA,10-K,373320,208,True,True,nvda-20260125 0001045810 2026 FY false 362 460...
7,TSLA,10-K,444095,247,True,True,tsla-20251231 false 0001318605 2025 FY http://...
8,V,10-Q,109473,61,True,True,v-20251231 0001403161 9/30 2026 Q1 false 50 50...


## 5. FRED Macro Layer For Economic Weather

Fetch the macro series that help describe the broader economic backdrop.


In [ ]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')
fred_series = {
    'FEDFUNDS': 'Fed Funds Rate',
    'CPIAUCSL': 'CPI',
    'UNRATE': 'Unemployment Rate',
    'DGS10': '10Y Treasury',
    'DGS2': '2Y Treasury',
}

fred_frames = []
for series_id, title in fred_series.items():
    fred_url = (
        'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    )
    payload = fetch_json(fred_url)
    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]
    frame['series_id'] = series_id
    frame['title'] = title
    fred_frames.append(frame)

fred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()
fred_df.to_csv(EXTERNAL_DIR / 'fred' / 'macro_series_full_review.csv', index=False)
fred_df.tail(20)


## 6. FMP Company Profile Enrichment For Full Universe

Use the current stable FMP profile endpoint to enrich the full ticker universe with sector, industry, market cap, and instrument type hints.


In [ ]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')
profile_frames = []
for symbol in ticker_universe:
    fmp_url = f'https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={FMP_API_KEY}'
    profile_payload = fetch_json(fmp_url)
    profile_frames.append(pd.DataFrame(profile_payload if isinstance(profile_payload, list) else [profile_payload]))

fmp_profiles = pd.concat(profile_frames, ignore_index=True) if profile_frames else pd.DataFrame()
fmp_profiles.to_csv(EXTERNAL_DIR / 'fmp' / 'company_profiles_full_universe.csv', index=False)
display_columns = [col for col in ['symbol', 'companyName', 'sector', 'industry', 'marketCap', 'isEtf', 'isFund'] if col in fmp_profiles.columns]
fmp_profiles[display_columns].head(20)


## 7. GDELT News Metadata For Full Universe

Collect a manageable number of recent article candidates per ticker. This is still metadata-first; we will inflate only a smaller set of URLs in the next step.


In [ ]:
news_frames = []
for symbol in ticker_universe:
    gdelt_url = (
        'https://api.gdeltproject.org/api/v2/doc/doc'
        f'?query={quote(symbol)}&mode=artlist&maxrecords={MAX_GDELT_ARTICLES_PER_TICKER}&format=json'
    )
    payload = fetch_json(gdelt_url)
    articles = pd.DataFrame(payload.get('articles', []))
    if articles.empty:
        continue
    articles['ticker'] = symbol
    news_frames.append(articles)

gdelt_articles = pd.concat(news_frames, ignore_index=True) if news_frames else pd.DataFrame()
gdelt_articles.to_csv(EXTERNAL_DIR / 'gdelt' / 'articles_full_universe.csv', index=False)
gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']].head(20) if not gdelt_articles.empty else pd.DataFrame()


## 8. Inflate News URLs And Review Usefulness

This is a practical review step: fetch a small sample of actual article pages, extract text, and inspect whether they are good enough for narrative understanding.


In [ ]:
article_reviews = []
if not gdelt_articles.empty:
    unique_articles = gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']].dropna(subset=['url']).drop_duplicates(subset=['url']).head(MAX_NEWS_URLS_TO_INFLATE)
    for row in unique_articles.itertuples(index=False):
        try:
            html = fetch_text(row.url)
            text = html_to_text(html)
            preview = simple_chunk(text, chunk_size=1200)
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'char_count': len(text),
                'contains_market_words': any(word in text.lower() for word in ['guidance', 'earnings', 'tariff', 'war', 'inflation', 'rate']),
                'text_preview': preview[0][:1200] if preview else '',
            })
        except Exception as error:
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'char_count': 0,
                'contains_market_words': False,
                'text_preview': f'ERROR: {error}',
            })

article_review_df = pd.DataFrame(article_reviews)
article_review_df.to_csv(REVIEW_DIR / 'gdelt_article_review.csv', index=False)
article_review_df.head(20) if not article_review_df.empty else pd.DataFrame()

## 9. Source Usefulness Review

This final table turns the source-review exercise into a product-facing judgment: what each source is good for, where it is weak, and whether it belongs in the structured layer or the vector layer.


In [ ]:
source_review = pd.DataFrame([
    {
        'source_name': 'SEC filing metadata',
        'artifact': 'latest_reporting_filings',
        'storage_layer': 'structured',
        'best_for': 'Official filing inventory, dates, forms, and primary document URLs',
        'weakness': 'Not insight by itself; mostly pointers and metadata',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC company facts',
        'artifact': 'company_facts_review',
        'storage_layer': 'structured',
        'best_for': 'Official financial facts and future feature engineering',
        'weakness': 'Messy taxonomy; needs interpretation and cleaning',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC filing text',
        'artifact': 'sec_doc_review_df',
        'storage_layer': 'vector',
        'best_for': 'Risk factors, business narrative, MD&A, and explainability',
        'weakness': 'Raw HTML and XBRL noise require cleanup',
        'product_value': 'Very high',
    },
    {
        'source_name': 'FRED macro data',
        'artifact': 'fred_df',
        'storage_layer': 'structured',
        'best_for': 'Economic weather and macro regime context',
        'weakness': 'Not company-specific',
        'product_value': 'High',
    },
    {
        'source_name': 'FMP profiles',
        'artifact': 'fmp_profiles',
        'storage_layer': 'structured',
        'best_for': 'Sector, industry, market cap, and instrument metadata',
        'weakness': 'Convenience enrichment, not source-of-truth',
        'product_value': 'Medium to high',
    },
    {
        'source_name': 'GDELT article metadata',
        'artifact': 'gdelt_articles',
        'storage_layer': 'structured',
        'best_for': 'Narrative discovery and URL inventory',
        'weakness': 'Metadata alone is shallow and noisy',
        'product_value': 'Medium',
    },
    {
        'source_name': 'Inflated news article text',
        'artifact': 'article_review_df',
        'storage_layer': 'vector',
        'best_for': 'Theme extraction, event-risk awareness, and market context',
        'weakness': 'Extraction quality varies widely by source and page template',
        'product_value': 'Medium to high',
    },
])
source_review.to_csv(REVIEW_DIR / 'source_usefulness_review.csv', index=False)
source_review
